In [0]:
%run ../lib/ingestion_functions

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from delta.tables import DeltaTable
import sys

In [0]:
# container_source = getArgument("container_source")
# directory_source = getArgument("directory_source")
# subdirectory_source = getArgument("subdirectory_source")
# file_source = getArgument("file_source")
# container_target = getArgument("container_target")
# directory_target = getArgument("directory_target")
# delta_table_name = getArgument("delta_table_name")
# id_field = getArgument("id_field")
# timestamp_field = getArgument("timestamp_field")

In [0]:
dbutils.widgets.text("container_source", "")
dbutils.widgets.text("directory_source", "")
dbutils.widgets.text("subdirectory_source", "")
dbutils.widgets.text("file_source", "")
dbutils.widgets.text("container_target", "")
dbutils.widgets.text("directory_target", "")
dbutils.widgets.text("delta_table_name", "")
dbutils.widgets.text("id_field", "")
dbutils.widgets.text("timestamp_field", "")

# Recuperando os valores
container_source = dbutils.widgets.get("container_source")
directory_source = dbutils.widgets.get("directory_source")
subdirectory_source = dbutils.widgets.get("subdirectory_source")
file_source = dbutils.widgets.get("file_source")
container_target = dbutils.widgets.get("container_target")
directory_target = dbutils.widgets.get("directory_target")
delta_table_name = dbutils.widgets.get("delta_table_name")
id_field = dbutils.widgets.get("id_field")
timestamp_field = dbutils.widgets.get("timestamp_field")


In [0]:
source_path = f"abfss://raw@adlsnovadriveeusdev.dfs.core.windows.net/{directory_source}/{subdirectory_source}/{file_source}/"
target_path = f"abfss://bronze@adlsnovadriveeusdev.dfs.core.windows.net/novadrive/incremental_load/{delta_table_name}"
schema_location = f"abfss://metastore@adlsnovadriveeusuc.dfs.core.windows.net/uc-metastore-novadrive-eus/dev/{delta_table_name}_schema"
checkpoint_location = f"abfss://metastore@adlsnovadriveeusuc.dfs.core.windows.net/uc-metastore-novadrive-eus/dev/{delta_table_name}_chk"

In [0]:
table_exists = spark.catalog.tableExists(f"dev.bronze.{delta_table_name}")
if not table_exists: # or subdirectory_source == "full_load":

    dbutils.fs.rm(checkpoint_location, True)
    dbutils.fs.rm(schema_location, True)

    creator = DeltaTableCreator(spark)

    bronzeInfo = creator.load(source_path, 'parquet')

    creator.create_table_with_cdf(
    df=bronzeInfo,
    catalog="dev",
    schema=container_target,
    table=delta_table_name,
    path=target_path,
    mode="overwrite"  # sobrescreve dados no path se já existirem
)    
elif table_exists and subdirectory_source == "full_load":

    dbutils.fs.rm(checkpoint_location, True)
    dbutils.fs.rm(schema_location, True)

    ingestor = Ingestor(
    spark=spark,
    source_path=source_path,
    data_format="parquet",
    target_path=target_path,
    catalog="dev",
    schemaname=container_target,
    tablename=delta_table_name
)

    
    print(f"Tabela {delta_table_name} existe e a carga é Full Load, executando...")

    ingestor.executeLoadAndSave(source_path)
    
else:

    print(f"Table {delta_table_name} already exists, doing the upsert...")

    incrementalIngestor = IncrementalIngestor(
        spark=spark,
        source_path=source_path,
        data_format="parquet",
        target_path=target_path,
        schema_location=schema_location,
        checkpoint_location=checkpoint_location,
        catalog="dev",
        schemaname="bronze",
        id_field= id_field,
        timestamp_field= timestamp_field,
        tablename=delta_table_name
     )
    incrementalIngestor.executeLoadAndSave(source_path)